In [ ]:
import copy
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import optuna

from DataProcessing import time_gaps


In [ ]:
SEQ_LEN = 128
STRIDE = 1
INPUT_DIM = 51

BATCH_SIZE = 512
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
CONV_CHANNELS = 64
KERNEL_SIZE = 3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
PATIENCE = 8

ATTACK_CONTEXT_PAD_SECONDS = 15 * 60
CV_FOLDS = 6
OPTUNA_TRIALS = 50
OPTUNA_TARGET = "macro_attack_f1"
RANDOM_SEED = 42

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)
print("device:", device)


#### Data processing

In [ ]:
def build_attack_segments(df_attack: pd.DataFrame, context_pad_seconds: int):
    attack_only = df_attack.loc[df_attack["Label"] == 1, ["Timestamp", "Detailed_Label"]].copy()
    attack_only = attack_only.sort_values("Timestamp").reset_index(drop=True)

    is_new_segment = (
        (attack_only["Detailed_Label"] != attack_only["Detailed_Label"].shift(1))
        | (attack_only["Timestamp"].diff().dt.total_seconds().fillna(1) != 1)
    )
    attack_only["attack_segment_id"] = is_new_segment.cumsum().astype(int)

    segment_table = (
        attack_only.groupby(["attack_segment_id", "Detailed_Label"])
        .agg(
            start=("Timestamp", "min"),
            end=("Timestamp", "max"),
            rows=("Timestamp", "size"),
        )
        .reset_index()
        .sort_values("attack_segment_id")
        .reset_index(drop=True)
    )

    pad = pd.Timedelta(seconds=context_pad_seconds)
    global_start = df_attack["Timestamp"].min()
    global_end = df_attack["Timestamp"].max()
    context_starts = []
    context_ends = []

    for i in range(len(segment_table)):
        segment_start = segment_table.loc[i, "start"]
        segment_end = segment_table.loc[i, "end"]

        if i == 0:
            prev_midpoint = global_start
        else:
            prev_end = segment_table.loc[i - 1, "end"]
            prev_midpoint = prev_end + (segment_start - prev_end) / 2

        if i == len(segment_table) - 1:
            next_midpoint = global_end
        else:
            next_start = segment_table.loc[i + 1, "start"]
            next_midpoint = segment_end + (next_start - segment_end) / 2

        context_starts.append(max(segment_start - pad, prev_midpoint))
        context_ends.append(min(segment_end + pad, next_midpoint))

    segment_table["context_start"] = context_starts
    segment_table["context_end"] = context_ends
    return attack_only[["Timestamp", "attack_segment_id"]], segment_table


def assign_context_groups(df_attack: pd.DataFrame, segment_table: pd.DataFrame):
    df = df_attack.copy()
    df["context_group"] = 0

    for row in segment_table.itertuples(index=False):
        mask = (df["Timestamp"] >= row.context_start) & (df["Timestamp"] <= row.context_end)
        df.loc[mask, "context_group"] = int(row.attack_segment_id)

    attack_mask = df["Label"] == 1
    df.loc[attack_mask, "context_group"] = df.loc[attack_mask, "attack_segment_id"]
    return df


def make_window_arrays(
    df: pd.DataFrame,
    feature_cols,
    scaler: MinMaxScaler,
    seq_len: int,
    stride: int,
    drop_outside_context: bool = False,
):
    X = scaler.transform(df[feature_cols]).astype(np.float32)
    y = df["Label"].to_numpy(dtype=np.int64)
    timestamps = pd.to_datetime(df["Timestamp"]).to_numpy()
    ts_seconds = pd.to_datetime(pd.Series(timestamps)).astype("int64") // 10**9
    ts_seconds = ts_seconds.to_numpy()

    attack_segment_ids = df["attack_segment_id"].to_numpy(dtype=np.int64)
    context_groups = df["context_group"].to_numpy(dtype=np.int64)

    X_seq = []
    y_seq = []
    records = []
    skipped_windows = 0
    mixed_attack_windows = 0
    outside_context_windows = 0

    for start in range(0, len(X) - seq_len + 1, stride):
        end = start + seq_len
        if not np.all(np.diff(ts_seconds[start:end]) == 1):
            skipped_windows += 1
            continue

        attack_ids = np.unique(attack_segment_ids[start:end][attack_segment_ids[start:end] > 0])
        if len(attack_ids) > 1:
            mixed_attack_windows += 1
            continue

        label = int(y[start:end].max())
        center_idx = start + seq_len // 2

        if label == 1:
            attack_segment_id = int(attack_ids[0])
            context_group = attack_segment_id
        else:
            attack_segment_id = 0
            context_group = int(context_groups[center_idx])
            if drop_outside_context and context_group == 0:
                outside_context_windows += 1
                continue

        X_seq.append(X[start:end])
        y_seq.append(label)
        records.append({
            "window_start": timestamps[start],
            "window_end": timestamps[end - 1],
            "window_center": timestamps[center_idx],
            "label": label,
            "context_group": context_group,
            "attack_segment_id": attack_segment_id,
        })

    X_seq = np.asarray(X_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=np.int64)
    meta_df = pd.DataFrame(records)
    return X_seq, y_seq, meta_df, skipped_windows, mixed_attack_windows, outside_context_windows


def load_group_cv_data(start_time, drop_columns, context_pad_seconds):
    df_normal = pd.read_parquet("../Dataset/SWaT_Dataset_Normal_v1.parquet")
    df_attack = pd.read_parquet("../Dataset/SWaT_Dataset_Attack_v1.parquet")

    df_normal["Timestamp"] = pd.to_datetime(df_normal["Timestamp"])
    df_attack["Timestamp"] = pd.to_datetime(df_attack["Timestamp"])

    print("\n------------------------- Original Data -------------------------")
    print(f"Normal Data = {df_normal.shape}")
    print(f"Attack Data = {df_attack.shape}")
    print("\n------------------------- Processing ... -------------------------")

    keep_mask = df_normal["Timestamp"] >= start_time
    df_normal = df_normal.loc[keep_mask].reset_index(drop=True)
    skipped_rows = int((~keep_mask).sum())
    print(f"Normal data = {df_normal.shape}")
    print(f"Skip data = {skipped_rows}")

    segment_lookup, segment_table = build_attack_segments(df_attack, context_pad_seconds)
    df_attack = df_attack.merge(segment_lookup, on="Timestamp", how="left")
    df_attack["attack_segment_id"] = df_attack["attack_segment_id"].fillna(0).astype(int)
    df_attack = assign_context_groups(df_attack, segment_table)

    train_normal_df, tmp_normal_df = train_test_split(df_normal, train_size=0.8, shuffle=False)
    earlystop_normal_df, threshold_normal_df = train_test_split(tmp_normal_df, train_size=0.5, shuffle=False)

    train_gaps = time_gaps(train_normal_df)
    earlystop_gaps = time_gaps(earlystop_normal_df)
    threshold_gaps = time_gaps(threshold_normal_df)
    attack_gaps = time_gaps(df_attack)

    feature_cols = [
        col for col in df_attack.columns
        if col not in {"Timestamp", "Label", "Detailed_Label", "attack_segment_id", "context_group"}
    ]

    print(f"train_normal_df = {train_normal_df.shape}")
    print(f"earlystop_normal_df = {earlystop_normal_df.shape}")
    print(f"threshold_normal_df = {threshold_normal_df.shape}")
    print(f"attack_eval_df = {df_attack.shape}")
    print(f"attack segments = {len(segment_table)}")
    print(f"train gaps: {len(train_gaps)}")
    print(f"earlystop gaps: {len(earlystop_gaps)}")
    print(f"threshold gaps: {len(threshold_gaps)}")
    print(f"attack gaps: {len(attack_gaps)}")
    print("\nAttack Segment Summary")
    print(segment_table[["attack_segment_id", "Detailed_Label", "rows", "context_start", "context_end"]].to_string(index=False))

    return train_normal_df, earlystop_normal_df, threshold_normal_df, df_attack, segment_table, feature_cols


start_time = pd.to_datetime("2015-12-23 12:00:00")
drop_columns = ["Detailed_Label"]

train_normal_df, earlystop_normal_df, threshold_normal_df, attack_eval_df, segment_table, feature_cols = load_group_cv_data(
    start_time=start_time,
    drop_columns=drop_columns,
    context_pad_seconds=ATTACK_CONTEXT_PAD_SECONDS,
)

INPUT_DIM = len(feature_cols)
print("INPUT_DIM:", INPUT_DIM)


In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_normal_df[feature_cols])


def add_normal_context_columns(df: pd.DataFrame):
    local_df = df.copy()
    local_df["attack_segment_id"] = 0
    local_df["context_group"] = 0
    return local_df


X_train_seq, y_train_seq, train_window_df, train_skipped, train_mixed, train_outside = make_window_arrays(
    add_normal_context_columns(train_normal_df),
    feature_cols,
    scaler,
    SEQ_LEN,
    STRIDE,
    drop_outside_context=False,
)

X_val_seq, y_val_seq, val_window_df, val_skipped, val_mixed, val_outside = make_window_arrays(
    add_normal_context_columns(earlystop_normal_df),
    feature_cols,
    scaler,
    SEQ_LEN,
    STRIDE,
    drop_outside_context=False,
)

X_threshold_seq, y_threshold_seq, threshold_window_df, threshold_skipped, threshold_mixed, threshold_outside = make_window_arrays(
    add_normal_context_columns(threshold_normal_df),
    feature_cols,
    scaler,
    SEQ_LEN,
    STRIDE,
    drop_outside_context=False,
)

X_attack_eval_seq, y_attack_eval_seq, attack_window_df, attack_skipped, attack_mixed, attack_outside = make_window_arrays(
    attack_eval_df,
    feature_cols,
    scaler,
    SEQ_LEN,
    STRIDE,
    drop_outside_context=True,
)

attack_name_map = segment_table.set_index("attack_segment_id")["Detailed_Label"].to_dict()
attack_window_df["attack_name"] = attack_window_df["attack_segment_id"].map(attack_name_map).fillna("Normal")
attack_window_df["context_name"] = attack_window_df["context_group"].map(attack_name_map)

print("X_train_seq     :", X_train_seq.shape, "skipped:", train_skipped)
print("X_val_seq       :", X_val_seq.shape, "skipped:", val_skipped)
print("X_threshold_seq :", X_threshold_seq.shape, "skipped:", threshold_skipped)
print(
    "X_attack_eval_seq:", X_attack_eval_seq.shape,
    "positives:", int(y_attack_eval_seq.sum()),
    "skipped:", attack_skipped,
    "mixed:", attack_mixed,
    "outside:", attack_outside,
)
print("Attack groups covered:", attack_window_df["context_group"].nunique())
print("Positive windows per attack group")
print(attack_window_df.groupby("context_group")["label"].agg(["count", "sum"]).to_string())

# Free raw frames once all window arrays are ready.
del train_normal_df, earlystop_normal_df, threshold_normal_df, attack_eval_df
gc.collect()


In [ ]:
train_dataset = TensorDataset(torch.from_numpy(X_train_seq), torch.from_numpy(np.zeros(len(X_train_seq), dtype=np.int64)))
val_dataset = TensorDataset(torch.from_numpy(X_val_seq), torch.from_numpy(np.zeros(len(X_val_seq), dtype=np.int64)))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


#### Model architecture with optuna framework

In [ ]:
def get_search_space(trial):
    score_mode = trial.suggest_categorical("score_mode", ["mean", "max", "topk"])

    params = {
        "conv_channels": trial.suggest_categorical("conv_channels", [32, 64, 128, 256]),
        "kernel_size": trial.suggest_categorical("kernel_size", [3, 5, 7]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128, 256]),
        "num_layers": trial.suggest_int("num_layers", 1, 5),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "bidirectional": trial.suggest_categorical("bidirectional", [False, True]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True),
        "threshold_percentile": trial.suggest_float("threshold_percentile", 90.0, 99.5),
        "score_mode": score_mode,
    }

    if score_mode == "topk":
        params["top_k"] = trial.suggest_int("top_k", 3, 20)
    else:
        params["top_k"] = 5

    return params


In [ ]:
class ConvFeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, conv_channels: int, kernel_size: int, dropout: float):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, conv_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x)
        return x.transpose(1, 2)


class Encoder(nn.Module):
    def __init__(self, conv_channels: int, hidden_size: int, num_layers: int, dropout: float, bidirectional: bool):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional, # BiLSTM，Encoder 和 Decoder 的方向性需一樣，否則維度會報錯
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x):
        _, (hidden, cell) = self.lstm(x)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, conv_channels: int, hidden_size: int, num_layers: int, dropout: float, bidirectional: bool):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional, # BiLSTM
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # BiLSTM 的 hidden size 會翻倍 
        direction_factor = 2 if bidirectional else 1
        self.output_layer = nn.Linear(hidden_size * direction_factor, conv_channels)

    def forward(self, decoder_input, hidden, cell):
        decoded, _ = self.lstm(decoder_input, (hidden, cell))
        return self.output_layer(decoded)


class ReconstructionHead(nn.Module):
    def __init__(self, conv_channels: int, output_dim: int, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.proj = nn.Conv1d(conv_channels, output_dim, kernel_size=kernel_size, padding=padding)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.proj(x)
        return x.transpose(1, 2)


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        conv_channels: int,
        kernel_size: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool, # BiLSTM
    ):
        super().__init__()
        self.feature_extractor = ConvFeatureExtractor(input_dim, conv_channels, kernel_size, dropout)
        self.encoder = Encoder(conv_channels, hidden_size, num_layers, dropout, bidirectional=bidirectional)
        self.decoder = Decoder(conv_channels, hidden_size, num_layers, dropout, bidirectional=bidirectional)
        self.reconstruction_head = ReconstructionHead(conv_channels, input_dim, kernel_size)

    def forward(self, x):
        # 特徵提取
        conv_features = self.feature_extractor(x)
        # Encoder 壓縮
        hidden, cell = self.encoder(conv_features)
        # Decoder 重建
        decoder_input = torch.zeros_like(conv_features)
        decoded_features = self.decoder(decoder_input, hidden, cell)
        # 重建
        reconstruction = self.reconstruction_head(decoded_features)
        return reconstruction


In [ ]:
# model = LSTMAutoencoder(
#     input_dim=INPUT_DIM,
#     conv_channels=CONV_CHANNELS,
#     kernel_size=KERNEL_SIZE,
#     hidden_size=HIDDEN_SIZE,
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT,
#     bidirectional=False,
# ).to(device)
#
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# print(model)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for (batch_x, _) in loader:
        batch_x = batch_x.to(device)

        optimizer.zero_grad()
        reconstruction = model(batch_x)
        loss = criterion(reconstruction, batch_x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


def evaluate_reconstruction_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for (batch_x, _) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            loss = criterion(reconstruction, batch_x)
            total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
history = {"train_loss": [], "val_loss": []}


def compute_window_score(reconstruction, batch_x, mode="mean", top_k=5):
    point_error = (reconstruction - batch_x) ** 2
    timestep_error = point_error.mean(dim=2)

    if mode == "mean":
        return timestep_error.mean(dim=1)
    if mode == "max":
        return timestep_error.max(dim=1).values
    if mode == "topk":
        k = min(top_k, timestep_error.shape[1])
        topk_vals = torch.topk(timestep_error, k=k, dim=1).values
        return topk_vals.mean(dim=1)

    raise ValueError(f"Unsupported mode: {mode}")


def get_reconstruction_errors(model, data_array: np.ndarray, device, batch_size: int = 512, score_mode: str = "mean", top_k: int = 5):
    model.eval()
    scores = []
    dataset = TensorDataset(torch.from_numpy(data_array).float())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            batch_error = compute_window_score(reconstruction, batch_x, mode=score_mode, top_k=top_k)
            scores.append(batch_error.detach().cpu().numpy())

    return np.concatenate(scores)


def fit_model(params, trial=None, capture_history=False):
    model = LSTMAutoencoder(
        input_dim=INPUT_DIM,
        conv_channels=params["conv_channels"],
        kernel_size=params["kernel_size"],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        bidirectional=params["bidirectional"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"])

    local_history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state_dict = None
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate_reconstruction_loss(model, val_loader, criterion, device)

        if capture_history:
            local_history["train_loss"].append(train_loss)
            local_history["val_loss"].append(val_loss)

        print(f"Epoch [{epoch:02d}/{EPOCHS}] train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

        if trial is not None:
            trial.report(best_val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    model.load_state_dict(best_state_dict)
    return model, best_val_loss, local_history


def compute_binary_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def evaluate_attack_group_cv(model, params):
    threshold_percentile = params["threshold_percentile"] / 100.0
    score_mode = params["score_mode"]
    top_k = params["top_k"]

    threshold_errors = get_reconstruction_errors(
        model,
        X_threshold_seq,
        device,
        batch_size=BATCH_SIZE,
        score_mode=score_mode,
        top_k=top_k,
    )
    threshold = float(np.quantile(threshold_errors, threshold_percentile))

    attack_errors = get_reconstruction_errors(
        model,
        X_attack_eval_seq,
        device,
        batch_size=BATCH_SIZE,
        score_mode=score_mode,
        top_k=top_k,
    )

    evaluation_df = attack_window_df.copy()
    evaluation_df["error"] = attack_errors
    evaluation_df["y_pred"] = (evaluation_df["error"] > threshold).astype(int)

    splitter = GroupKFold(n_splits=CV_FOLDS)
    dummy_X = np.zeros(len(evaluation_df))
    group_to_fold = {}
    fold_rows = []

    for fold_idx, (_, test_idx) in enumerate(
        splitter.split(dummy_X, evaluation_df["label"], groups=evaluation_df["context_group"]),
        start=1,
    ):
        fold_df = evaluation_df.iloc[test_idx].copy()
        fold_groups = sorted(fold_df["context_group"].unique().tolist())
        for group_id in fold_groups:
            group_to_fold[group_id] = fold_idx

        fold_metrics = compute_binary_metrics(fold_df["label"], fold_df["y_pred"])
        fold_rows.append({
            "fold": fold_idx,
            "n_attacks": len(fold_groups),
            "n_windows": len(fold_df),
            "n_positive_windows": int(fold_df["label"].sum()),
            "window_precision": fold_metrics["precision"],
            "window_recall": fold_metrics["recall"],
            "window_f1": fold_metrics["f1"],
            "attack_segments": ", ".join(attack_name_map[group_id] for group_id in fold_groups),
        })

    per_attack_rows = []
    for attack_id, group_df in evaluation_df.groupby("context_group"):
        attack_metrics = compute_binary_metrics(group_df["label"], group_df["y_pred"])
        per_attack_rows.append({
            "attack_segment_id": int(attack_id),
            "attack_name": attack_name_map[int(attack_id)],
            "fold": group_to_fold[int(attack_id)],
            "n_windows": len(group_df),
            "n_positive_windows": int(group_df["label"].sum()),
            "precision": attack_metrics["precision"],
            "recall": attack_metrics["recall"],
            "f1": attack_metrics["f1"],
        })

    per_attack_df = pd.DataFrame(per_attack_rows).sort_values("attack_segment_id").reset_index(drop=True)
    fold_df = pd.DataFrame(fold_rows)
    fold_macro_df = (
        per_attack_df.groupby("fold")[["precision", "recall", "f1"]]
        .mean()
        .rename(columns={
            "precision": "macro_attack_precision",
            "recall": "macro_attack_recall",
            "f1": "macro_attack_f1",
        })
        .reset_index()
    )
    fold_df = fold_df.merge(fold_macro_df, on="fold", how="left")

    global_metrics = compute_binary_metrics(evaluation_df["label"], evaluation_df["y_pred"])

    return {
        "threshold": threshold,
        "threshold_errors": threshold_errors,
        "evaluation_df": evaluation_df,
        "fold_df": fold_df,
        "per_attack_df": per_attack_df,
        "global_metrics": global_metrics,
        "macro_attack_precision": float(per_attack_df["precision"].mean()),
        "macro_attack_recall": float(per_attack_df["recall"].mean()),
        "macro_attack_f1": float(per_attack_df["f1"].mean()),
        "cv_window_precision": float(fold_df["window_precision"].mean()),
        "cv_window_recall": float(fold_df["window_recall"].mean()),
        "cv_window_f1": float(fold_df["window_f1"].mean()),
        "global_window_precision": global_metrics["precision"],
        "global_window_recall": global_metrics["recall"],
        "global_window_f1": global_metrics["f1"],
    }


def run_experiment(params, trial=None, capture_history=False):
    model, best_val_loss, local_history = fit_model(params, trial=trial, capture_history=capture_history)
    cv_results = evaluate_attack_group_cv(model, params)
    return model, best_val_loss, local_history, cv_results


def objective(trial):
    params = get_search_space(trial)
    _, best_val_loss, _, cv_results = run_experiment(params, trial=trial, capture_history=False)
    score = cv_results[OPTUNA_TARGET]

    trial.set_user_attr("best_val_loss", best_val_loss)
    trial.set_user_attr("threshold", cv_results["threshold"])
    trial.set_user_attr("macro_attack_precision", cv_results["macro_attack_precision"])
    trial.set_user_attr("macro_attack_recall", cv_results["macro_attack_recall"])
    trial.set_user_attr("macro_attack_f1", cv_results["macro_attack_f1"])
    trial.set_user_attr("cv_window_f1", cv_results["cv_window_f1"])
    trial.set_user_attr("global_window_f1", cv_results["global_window_f1"])

    return score


In [ ]:
sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=OPTUNA_TRIALS)

best_params = study.best_params

print(f"Best {OPTUNA_TARGET}: {study.best_value:.4f}")
print("Best params:", best_params)
print("Best attrs:", study.best_trial.user_attrs)


In [ ]:
best_model, best_val_loss, history, cv_results = run_experiment(best_params, capture_history=True)

threshold = cv_results["threshold"]
threshold_errors = cv_results["threshold_errors"]
evaluation_df = cv_results["evaluation_df"]
fold_df = cv_results["fold_df"]
per_attack_df = cv_results["per_attack_df"]
attack_errors = evaluation_df["error"].to_numpy()
y_true = evaluation_df["label"].to_numpy()
y_pred = evaluation_df["y_pred"].to_numpy()

print(f"Best {OPTUNA_TARGET}: {study.best_value:.4f}")
print(f"Best val loss: {best_val_loss:.6f}")
print(f"Threshold: {threshold:.6f}")
print(f"Macro attack precision: {cv_results['macro_attack_precision']:.4f}")
print(f"Macro attack recall   : {cv_results['macro_attack_recall']:.4f}")
print(f"Macro attack F1       : {cv_results['macro_attack_f1']:.4f}")
print(f"CV window precision   : {cv_results['cv_window_precision']:.4f}")
print(f"CV window recall      : {cv_results['cv_window_recall']:.4f}")
print(f"CV window F1          : {cv_results['cv_window_f1']:.4f}")
print(f"Global window F1      : {cv_results['global_window_f1']:.4f}")


#### Experimental Result

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss value")
plt.title("Training Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
fold_df[[
    "fold",
    "n_attacks",
    "n_windows",
    "n_positive_windows",
    "window_precision",
    "window_recall",
    "window_f1",
    "macro_attack_precision",
    "macro_attack_recall",
    "macro_attack_f1",
    "attack_segments",
]].round(4)


In [ ]:
per_attack_df[[
    "attack_segment_id",
    "attack_name",
    "fold",
    "n_windows",
    "n_positive_windows",
    "precision",
    "recall",
    "f1",
]].round(4)


In [ ]:
positive_errors = evaluation_df.loc[evaluation_df["label"] == 1, "error"]
negative_errors = evaluation_df.loc[evaluation_df["label"] == 0, "error"]

plt.figure(figsize=(10, 4))
plt.hist(threshold_errors, bins=60, alpha=0.6, label="threshold_normal_errors")
plt.hist(negative_errors, bins=60, alpha=0.5, label="attack_context_normal_errors")
plt.hist(positive_errors, bins=60, alpha=0.5, label="attack_context_positive_errors")
plt.axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.6f}")
plt.xlabel("Reconstruction Error")
plt.ylabel("Count")
plt.title("Error Distribution")
plt.legend()
plt.show()


In [ ]:
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("\nClassification Report")
print(classification_report(y_true, y_pred, digits=4, zero_division=0))
print("Confusion Matrix:\n", cm)


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Global Window Confusion Matrix")
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(attack_errors, label="reconstruction_error", linewidth=1)
plt.plot(y_true * threshold, label="true_anomaly(label scaled)", alpha=0.8)
plt.axhline(threshold, color="red", linestyle="--", label="threshold")
plt.xlabel("Attack Evaluation Window Index")
plt.ylabel("Error")
plt.title("Attack Context Reconstruction Error")
plt.legend()
plt.show()
